In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd
import glob

path = '/content/drive/MyDrive/idx_dataset/'
files = glob.glob(path + 'cleaned_df.csv')

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(df.shape)
df.head()

(106857, 101)


,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,LivingArea,...,AssociationFeeFrequency_Monthly,AssociationFeeFrequency_None,AssociationFeeFrequency_Quarterly,AssociationFeeFrequency_SemiAnnually,StateOrProvince_CA,StateOrProvince_FL,StateOrProvince_VA,Stories_2.0,Stories_Unknown,CloseYearMonth
0,1,0,0,0,1243810.0,1145272633,1243810.0,34.120270,-118.232010,1642.0,...,True,False,False,False,True,False,False,False,True,202510
1,1,0,0,0,1227060.0,1145272243,1227060.0,34.119385,-118.233154,1680.0,...,True,False,False,False,True,False,False,False,True,202510
2,0,0,0,0,2708000.0,1145270695,2708000.0,37.514001,-122.001594,3179.0,...,False,True,False,False,True,False,False,False,True,202510
3,1,0,0,0,998000.0,1145264774,1050000.0,37.804249,-121.948020,1586.0,...,True,False,False,False,True,False,False,True,False,202510
4,0,0,0,0,2400000.0,1145259656,2400000.0,34.071070,-118.274262,6523.0,...,False,True,False,False,True,False,False,False,True,202510


In [15]:
df.columns.tolist()

['ViewYN',
 'WaterfrontYN',
 'BasementYN',
 'PoolPrivateYN',
 'OriginalListPrice',
 'ListingKey',
 'ClosePrice',
 'Latitude',
 'Longitude',
 'LivingArea',
 'ListPrice',
 'DaysOnMarket',
 'ListingKeyNumeric',
 'AttachedGarageYN',
 'ParkingTotal',
 'LotSizeAcres',
 'YearBuilt',
 'StreetNumberNumeric',
 'BathroomsTotalInteger',
 'BedroomsTotal',
 'FireplaceYN',
 'LotSizeArea',
 'MainLevelBedrooms',
 'NewConstructionYN',
 'GarageSpaces',
 'AssociationFee',
 'LotSizeSquareFeet',
 'ListingContractDate_year',
 'ListingContractDate_month',
 'PurchaseContractDate_year',
 'PurchaseContractDate_month',
 'ContractStatusChangeDate_year',
 'ContractStatusChangeDate_month',
 'CloseDate_year',
 'CloseDate_month',
 'BuyerOfficeName_freq',
 'ListOfficeName_freq',
 'SubdivisionName_freq',
 'CoListOfficeName_freq',
 'PostalCode_freq',
 'ElementarySchool_freq',
 'MLSAreaMajor_freq',
 'City_freq',
 'MiddleOrJuniorSchool_freq',
 'HighSchool_freq',
 'HighSchoolDistrict_freq',
 'Flooring_freq',
 'CountyOrParis

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("df shape:", df.shape)

target = "ClosePrice"

# 1. 排除：ListPrice / OriginalListPrice
#   - off-market房产没有这两个字段，模型若依赖它们就无法用于场外估价
#   - 二者与ClosePrice高度相关（挂牌价已隐含agent对成交价的预判），
#     属于target leakage：会让模型在train/test上表现"虚高"，
#     但学到的其实是复现挂牌定价逻辑，而非房产本身价值规律
REQUIRED_DROP = ["ListPrice", "OriginalListPrice"]

# 2. 排除：与REQUIRED_DROP同性质的leakage / 无意义列
# 这些不是任务里明确点名的，但逻辑上和ListPrice/OriginalListPrice是同一类问题：
#   - ListingKey / ListingKeyNumeric: 纯ID，无预测价值
#   - DaysOnMarket: 只有挂牌后才存在，off-market房产查询时不存在，
#                   而且和ClosePrice强相关（挂牌越久可能降价越多）
#   - PurchaseContractDate_year/month, ContractStatusChangeDate_year/month,
#     CloseDate_year/month, CloseYearMonth:
#     这些都是成交流程中、甚至成交之后才产生的时间信息，
#     本质上和ClosePrice同期发生，用于预测ClosePrice是时间穿越型leakage
SUGGESTED_DROP = [
    "ListingKey",
    "ListingKeyNumeric",
    "DaysOnMarket",
    "PurchaseContractDate_year", "PurchaseContractDate_month",
    "ContractStatusChangeDate_year", "ContractStatusChangeDate_month",
    "CloseDate_year", "CloseDate_month",
    "CloseYearMonth",
]

REDUNDANT_LOT_COLS = ["LotSizeAcres", "LotSizeArea"]  # Keep LotSizeSquareFeet only

DROP_COLS = REQUIRED_DROP + SUGGESTED_DROP + REDUNDANT_LOT_COLS

drop_cols_present = [c for c in DROP_COLS if c in df.columns]
missing_expected = [c for c in DROP_COLS if c not in df.columns]
print("Cols to be dropped:", drop_cols_present)
if missing_expected:
    print("Not found in dataset:", missing_expected)

df_model = df.drop(columns=drop_cols_present)





df shape: (106857, 101)
Cols to be dropped: ['ListPrice', 'OriginalListPrice', 'ListingKey', 'ListingKeyNumeric', 'DaysOnMarket', 'PurchaseContractDate_year', 'PurchaseContractDate_month', 'ContractStatusChangeDate_year', 'ContractStatusChangeDate_month', 'CloseDate_year', 'CloseDate_month', 'CloseYearMonth', 'LotSizeAcres', 'LotSizeArea']


In [17]:
# 3. 特征 / 目标 拆分
assert target in df_model.columns, f"{target} Not in df, check the name"

X = df_model.drop(columns=[target])
y = df_model[target]

# Check：ListPrice/OriginalListPrice not in columns
assert "ListPrice" not in X.columns and "OriginalListPrice" not in X.columns, \
    "ListPrice/OriginalListPrice still in cols"

# 4. Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

Train set: (85485, 86), Test set: (21372, 86)


In [18]:
# 5. Train baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# 6. Evaluation
y_pred = baseline_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n===== Baseline Linear Regression Result =====")
print(f"R2 (test):   {r2:.4f}")
print(f"MAE (test):  {mae:,.0f}")
print(f"RMSE (test): {rmse:,.0f}")

# 7. Record baseline result
model_results = pd.DataFrame([{
    "Model": "Linear Regression (Baseline)",
    "N_Features": X.shape[1],
    "Excluded_Cols": ", ".join(drop_cols_present),
    "R2_test": r2,
    "MAE_test": mae,
    "RMSE_test": rmse,
    "Notes": "Week4 baseline; excludes ListPrice/OriginalListPrice (task requirement) "
             "and other post-listing/transaction-time leakage columns"
}])

model_results


===== Baseline Linear Regression 结果 =====
R2 (test):   0.5434
MAE (test):  350,618
RMSE (test): 516,461


,Model,N_Features,Excluded_Cols,R2_test,MAE_test,RMSE_test,Notes
0,Linear Regression (Baseline),86,"ListPrice, OriginalListPrice, ListingKey, List...",0.543448,350617.554103,516460.600915,Week4 baseline; excludes ListPrice/OriginalLis...


In [21]:
# Make sure it is not overfitting
train_r2 = r2_score(y_train, baseline_model.predict(X_train))
print(f"R2 (train): {train_r2:.4f}")
print(f"R2 (test):  {r2:.4f}")

R2 (train): 0.5634
R2 (test):  0.5434


In [22]:
# Take a look at the coefficients
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": baseline_model.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df.head(15))
print(coef_df.tail(10))

                          feature   coefficient
62     Levels_One,Two,ThreeOrMore  2.339176e+06
83             StateOrProvince_VA  8.426175e+05
76  PropertyType_ResidentialLease  7.628665e+05
81             StateOrProvince_CA  6.721101e+05
61      Levels_One,Two,MultiSplit -4.665969e+05
56          Levels_MultiSplit,One -4.418816e+05
60                 Levels_One,Two -4.261577e+05
70         Levels_Two,ThreeOrMore -4.153035e+05
69                 Levels_Two,One -4.111423e+05
72                 Levels_Unknown  3.885841e+05
67                     Levels_Two -3.428638e+05
47     PropertySubType_MobileHome  3.116982e+05
55        PropertySubType_Unknown -3.024034e+05
1                    WaterfrontYN  2.855739e+05
39    PropertySubType_CoOwnership  2.821967e+05
                                  feature   coefficient
24                  CoListOfficeName_freq -9.734435e-01
29              MiddleOrJuniorSchool_freq -9.535890e-01
26                  ElementarySchool_freq -8.303682e-01
32      

In [59]:
# 1. 挂载Google Drive（如果还没挂载）
from google.colab import drive
drive.mount('/content/drive')

# 2. 输入token（不会保存到文件）
import getpass
token = getpass.getpass("输入你的GitHub Token: ")

!git remote set-url origin https://{token}@github.com/rongweishen63-blip/idx-exchange-project.git


# 3. Clone repo
%cd /content
!git clone https://{token}@github.com/rongweishen63-blip/idx-exchange-project.git
%cd idx-exchange-project

# 4. 复制notebook
!cp "/content/drive/MyDrive/Colab Notebooks/idx_project.ipynb" .

# 5. Push
!git config --global user.email "rongweishen63@gmail.com"
!git config --global user.name "rongweishen63-blip"

!cp "/content/drive/MyDrive/Colab Notebooks/03_baseline_model.ipynb" .

!git add .
!git commit -m "baseline"
!git push origin main

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: Interrupted by user